In [ ]:
# Import the RRD helper module in this sample folder
from pathlib import Path
import sys
import importlib

cwd = Path.cwd().resolve()
if (cwd / "rrd_helper.py").exists():
    SAMPLE_DIR = cwd
elif (cwd / "data" / "sample" / "mitake_error_log" / "rrd_helper.py").exists():
    SAMPLE_DIR = cwd / "data" / "sample" / "mitake_error_log"
else:
    SAMPLE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path(".").resolve()

sys.path.insert(0, str(SAMPLE_DIR))
import rrd_helper
importlib.reload(rrd_helper)
from rrd_helper import run_rrdtool, fetch_rrd, info_rrd
print(f"RRD Helper loaded from: {SAMPLE_DIR}")


## Course architecture boundary

本課程有兩條資料路徑，請不要混在一起：

- **Actual operating path**: OS / network signals -> exporter -> Prometheus -> Grafana.
- **Python learning path**: organized telemetry CSV or previous notebook output -> Python analysis -> result CSV -> `outputs/prometheus-dropzone/current_results.csv` -> `python_results_exporter` -> Prometheus -> Grafana.

除非 cell 明確寫的是 operational deployment example，Python anomaly / forecasting / RCA notebooks 的主要輸入是 organized telemetry CSV 或前一個 notebook 的輸出，不是 Prometheus query。Grafana 也不直接讀 CSV；Grafana 查 Prometheus。


In [ ]:
import datetime
from pathlib import Path

# 替換成您的 RRD 檔案路徑
rrd_file = SAMPLE_DIR / '資訊部/LibreNMS網路監控/Firewall_RRD_Log(Triggered off)/port-id7431.rrd'

try:
    # Verify file exists
    rrd_path = Path(rrd_file)
    if not rrd_path.exists():
        raise FileNotFoundError(f"RRD file not found: {rrd_file}")
    
    # ==========================================
    # 1. 取得檔案資訊
    # ==========================================
    info = info_rrd(str(rrd_path))
    print("📊 RRD File Information:")
    for key, value in list(info.items())[:5]:  # Show first 5 items
        print(f"  {key}: {value}")
    
    # Extract last update time from info
    if 'last_update' in info:
        last_timestamp = int(info['last_update'])
        readable_time = datetime.datetime.fromtimestamp(last_timestamp).strftime('%Y-%m-%d %H:%M:%S')
        print(f"\n✅ 檔案最後更新時間為: {readable_time} (Timestamp: {last_timestamp})")
    else:
        print("⚠️  Could not find last_update in RRD info")

    # ==========================================
    # 2. 設定抓取範圍 (例如：往前推 24 小時)
    # ==========================================
    start_timestamp = last_timestamp - 86400  # 86400 秒 = 24 小時
    end_timestamp = last_timestamp

    print(f"⏳ 正在抓取 {start_timestamp} 到 {end_timestamp} 的資料...\n")

    # ==========================================
    # 3. 抓取資料
    # ==========================================
    import importlib
    import rrd_helper
    importlib.reload(rrd_helper)
    from rrd_helper import fetch_rrd

    result = fetch_rrd(str(rrd_path), 'AVERAGE', start_timestamp, end_timestamp)
    
    columns = result['header'].get('columns', [])
    data_rows = result['data']
    step = None
    if len(data_rows) > 1:
        step = data_rows[1]['timestamp'] - data_rows[0]['timestamp']
    
    # ==========================================
    # 4. 格式化並印出結果
    # ==========================================
    print("-" * 50)
    print(f"資料來源欄位 (Data Sources): {columns}")
    if step is not None:
        print(f"資料解析度 (Step): 每 {step} 秒一筆")
    else:
        print("資料解析度 (Step): 無法自動計算")
    print("-" * 50)

    # 這裡我們只印出前 15 筆作為確認
    for i, row in enumerate(data_rows[:15]):
        timestamp = row['timestamp']
        values = row['values']
        
        # 將當下的 Unix Timestamp 轉為易讀時間
        dt = datetime.datetime.fromtimestamp(timestamp).strftime('%Y-%m-%d %H:%M:%S')

        # 格式化數值
        formatted_row = [f"{val:.4f}" if val is not None else "NaN" for val in values]

        print(f"{dt} : {formatted_row}")

except FileNotFoundError as e:
    print(f"❌ 檔案未找到: {e}")
except RuntimeError as e:
    print(f"❌ RRD 操作失敗: {e}")
except Exception as e:
    print(f"❌ 發生未預期的錯誤: {e}")
